In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from lightgbm import LGBMRegressor

In [2]:
## featured data from parquet file 

featured_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "featured_data.parquet"
)
data = pd.read_parquet(featured_PATH)

data

,date,consumption,year,month,day,weekday,national_temp,hour,consumption_lag_24,consumption_lag_48,...,CDD,temp_lag_24,temp_lag_48,temp_rolling_mean_24,yearly_baseline,target_ratio,day_length,is_dark,hour_to_sunrise,is_covid
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,8.166667,0
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,7.166667,0
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,6.166667,0
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,5.166667,0
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,4.166667,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,...,9.720893,23.592781,23.277993,21.967974,40589.056963,1.071402,13.166667,0,NaN,0
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,...,8.377825,22.749788,22.444577,21.980691,40588.471465,1.091602,13.166667,1,10.233333,0
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,...,8.069101,22.351889,21.814356,22.003089,40587.751648,1.060576,13.166667,1,9.233333,0
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,...,7.356092,21.627122,20.642315,22.044122,40586.758430,1.023994,13.166667,1,8.233333,0


In [3]:
# Check NaN 

data.isna().sum()

# Traget variable is target_ratio thus it can not contain NaN
# Thus the first 8789 lines shoul be dropped. The remaining NaN are in hours_to_sunrise which are daylight data.

data = data.dropna(subset=["target_ratio"])
data.isna().sum()

date                                0
consumption                         0
year                                0
month                               0
day                                 0
weekday                             0
national_temp                       0
hour                                0
consumption_lag_24                  0
consumption_lag_48                  0
consumption_lag_168                 0
consumption_rolling_mean_24         0
consumption_rolling_mean_168        0
is_weekend                          0
DoW                                 0
DoY                                 0
is_religious_holiday                0
is_arife                            0
is_national_holiday                 0
is_kopru                            0
bayram_day_index                    0
HDD                                 0
CDD                                 0
temp_lag_24                         0
temp_lag_48                         0
temp_rolling_mean_24                0
yearly_basel

In [4]:
# ---------Train-Val-Test Split-------------

#---Split the data train val test---
train = data[data["date"] < "2024-09-01"].copy()
val   = data[(data["date"] >= "2024-09-01") & (data["date"] < "2025-09-01")].copy()
test  = data[data["date"] >= "2025-09-01"].copy()
print(len(train), len(val), len(test))


exclude_cols = ["date", "weekday", "consumption", "target_ratio", "day"]
feature_cols = [col for col in data.columns if col not in exclude_cols]

categorical_cols = ["hour", "DoW", "bayram_day_index", "DoY", "month"]


X_train, y_train = train[feature_cols], train["target_ratio"]
X_val, y_val = val[feature_cols], val["target_ratio"]

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")

58411 8760 8736


C:\Users\Sultan Selin\AppData\Local\Temp\ipykernel_7924\3773287743.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].astype("category")
C:\Users\Sultan Selin\AppData\Local\Temp\ipykernel_7924\3773287743.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val[col] = X_val[col].astype("category")


In [5]:
# -------- LightGBM Model -------------#
model = LGBMRegressor(
    objective = "regression",
    random_state=42,
    importance_type="gain",
    max_bin = 400 #to address DoY(366)
)

model.fit(X_train, y_train, categorical_feature=categorical_cols)
importance_gain = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)###
importance_df = importance_gain.reset_index()
importance_df.columns = ["feature", "gain"]
importance_df

c:\Users\Sultan Selin\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Sultan Selin\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Sultan Selin\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Sultan Selin\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executabl

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003785 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5889
[LightGBM] [Info] Number of data points in the train set: 58411, number of used features: 27
[LightGBM] [Info] Start training from score 1.013554


,feature,gain
0,consumption_lag_168,4106.926012
1,consumption_lag_24,1652.832910
2,yearly_baseline,606.668796
3,DoY,421.517196
4,is_religious_holiday,300.585488
5,DoW,186.537745
6,is_weekend,152.395557
7,hour,91.008007
8,CDD,73.707648
9,is_arife,33.313139


In [6]:
# ------Consumption calc over target ratio ----------#

predicted_ratio = model.predict(X_val)
predicted_consumption = predicted_ratio * val["yearly_baseline"]

MAPE = ((val["consumption"] - predicted_consumption).abs() / val["consumption"]).mean() * 100
print("LightGBM Val MAPE (%):", MAPE)


val_copy = val.copy()
val_copy["predicted_consumption"] = predicted_consumption
val_copy["abs_pct_error"] = ((val_copy["consumption"] - val_copy["predicted_consumption"]).abs() / val_copy["consumption"]) * 100

print("Hourly MAPE:", val_copy.groupby("hour")["abs_pct_error"].mean())


LightGBM Val MAPE (%): 4.219217380827702
Hourly MAPE: hour
0     4.456849
1     4.302535
2     4.049599
3     3.943476
4     3.938350
5     4.019800
6     4.005238
7     4.385392
8     4.744401
9     4.561368
10    4.530170
11    4.549686
12    4.488376
13    4.610115
14    4.741077
15    4.727863
16    4.550162
17    4.150584
18    3.679498
19    3.607258
20    3.591884
21    3.799092
22    3.865406
23    3.963039
Name: abs_pct_error, dtype: float64


In [7]:
predicted_ratio_train_v1 = model.predict(X_train)
predicted_consumption_train= predicted_ratio_train_v1 * train["yearly_baseline"]

MAPE_train = ((train["consumption"] - predicted_consumption_train).abs() / train["consumption"]).mean() * 100
print("LightGBM Train_v1 MAPE (%):", MAPE_train)

LightGBM Train_v1 MAPE (%): 1.217843901909221


In [8]:
print("Weekday MAPE:", val_copy.groupby("weekday")["abs_pct_error"].mean())


Weekday MAPE: weekday
Friday       4.095173
Monday       4.722314
Saturday     4.265625
Sunday       5.355654
Thursday     3.778365
Tuesday      3.594666
Wednesday    3.700869
Name: abs_pct_error, dtype: float64


In [9]:
##############--------- Model Version 2 ---------##############
# Drop some features based on importance gain, redundancy and relevancy

drop_features = ["month", "year", "is_dark", "temp_lag_48", "consumption_rolling_mean_24"]

feature_cols_v2 = [c for c in feature_cols if c not in drop_features]
categorical_cols_v2 = [c for c in categorical_cols if c not in drop_features]

X_train_v2 = X_train[feature_cols_v2]
X_val_v2 = X_val[feature_cols_v2]

model_v2 = LGBMRegressor(
    objective= "regression",
    random_state= 42,
    importance_type= "gain",
    max_bin = 400
)

model_v2.fit(X_train_v2, y_train, categorical_feature=categorical_cols_v2)

predicted_ratio_v2 = model_v2.predict(X_val_v2)
predicted_consumption_v2 = predicted_ratio_v2*val["yearly_baseline"]


MAPE_v2 = ((val["consumption"] - predicted_consumption_v2).abs() / val["consumption"]).mean() * 100
print("LightGBM v2 Val MAPE (%):", MAPE_v2)

importance_gain_v2 = pd.Series(model_v2.feature_importances_, index=feature_cols_v2).sort_values(ascending=False)
importance_df = importance_gain_v2.reset_index()
importance_df.columns = ["feature", "gain"]
importance_df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005117 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5066
[LightGBM] [Info] Number of data points in the train set: 58411, number of used features: 22
[LightGBM] [Info] Start training from score 1.013554
LightGBM v2 Val MAPE (%): 4.230689785466259


,feature,gain
0,consumption_lag_168,4110.522816
1,consumption_lag_24,1649.958693
2,yearly_baseline,605.787531
3,DoY,423.364884
4,is_religious_holiday,299.596216
5,DoW,189.205955
6,is_weekend,150.413865
7,hour,89.368772
8,CDD,73.401346
9,is_arife,34.436653


In [10]:
predicted_ratio_train_v2 = model_v2.predict(X_train_v2)
predicted_consumption_train_v2= predicted_ratio_train_v2 * train["yearly_baseline"]

MAPE_train = ((train["consumption"] - predicted_consumption_train_v2).abs() / train["consumption"]).mean() * 100
print("LightGBM Train_v2 MAPE (%):", MAPE_train)

LightGBM Train_v2 MAPE (%): 1.2105512073076201


In [11]:
##############--------- Model Version 3 ---------##############
# Drop some features based on importance gain, redundancy and relevancy
drop_features_v3 = ["is_kopru", "day_length", "consumption_rolling_mean_168"]

feature_cols_v3 = [c for c in feature_cols_v2 if c not in drop_features_v3]
categorical_cols_v3 = [c for c in categorical_cols_v2 if c not in drop_features_v3]

X_train_v3 = X_train[feature_cols_v3]
X_val_v3 = X_val[feature_cols_v3]

model_v3 = LGBMRegressor(
    objective="regression",
    random_state=42,
    importance_type="gain",
    max_bin = 400
)
model_v3.fit(X_train_v3, y_train, categorical_feature=categorical_cols_v3)

predicted_ratio_v3 = model_v3.predict(X_val_v3)
predicted_consumption_v3 = predicted_ratio_v3 * val["yearly_baseline"]

MAPE_v3 = ((val["consumption"] - predicted_consumption_v3).abs() / val["consumption"]).mean() * 100
print("LightGBM v3 Val MAPE (%):", MAPE_v3)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002244 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4412
[LightGBM] [Info] Number of data points in the train set: 58411, number of used features: 19
[LightGBM] [Info] Start training from score 1.013554
LightGBM v3 Val MAPE (%): 4.101010239913978


In [12]:
predicted_ratio_train_v3 = model_v3.predict(X_train_v3)
predicted_consumption_train_v3 = predicted_ratio_train_v3 * train["yearly_baseline"]

MAPE_train = ((train["consumption"] - predicted_consumption_train_v3).abs() / train["consumption"]).mean() * 100
print("LightGBM Train MAPE (%):", MAPE_train)

LightGBM Train MAPE (%): 1.2208908804417702


In [29]:
##########--------------HYPERPARAMETER TUNING--------------##########

import optuna

def objective(trial):
    params = {
        "objective": "regression",
        "random_state": 42,
        "verbose": -1,
        "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 300),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 255, 500),
    }

    model = LGBMRegressor(**params)
    model.fit(X_train_v3, y_train, categorical_feature=categorical_cols_v3)

    # evaluate in consumption space
    predicted_ratio = model.predict(X_val_v3)
    predicted_consumption = predicted_ratio * val["yearly_baseline"]
    return ((val["consumption"] - predicted_consumption).abs() / val["consumption"]).mean() * 100

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-10-01 18:27:13,690] A new study created in memory with name: no-name-5bd84e2b-4ab5-4958-a024-5c9826b121be
[I 2026-10-01 18:27:16,408] Trial 0 finished with value: 3.3988548849266396 and parameters: {'n_estimators': 385, 'learning_rate': 0.24837100260005374, 'num_leaves': 34, 'max_depth': 9, 'min_child_samples': 75, 'colsample_bytree': 0.5314171679977959, 'subsample': 0.8950398219352051, 'subsample_freq': 5, 'reg_alpha': 0.00996941010607, 'reg_lambda': 3.152572230099697e-05, 'max_bin': 450}. Best is trial 0 with value: 3.3988548849266396.
[I 2026-10-01 18:27:18,779] Trial 1 finished with value: 3.8207583481132454 and parameters: {'n_estimators': 269, 'learning_rate': 0.0734379936899741, 'num_leaves': 67, 'max_depth': 5, 'min_child_samples': 222, 'colsample_bytree': 0.7361377484520375, 'subsample': 0.9703510755151576, 'subsample_freq': 1, 'reg_alpha': 2.269359063846955, 'reg_lambda': 0.6941480763039936, 'max_bin': 426}. Best is trial 0 with value: 3.3988548849266396.
[I 2026-10-0

Best MAPE: 2.919485122730832
Best params: {'n_estimators': 813, 'learning_rate': 0.07420147247657288, 'num_leaves': 47, 'max_depth': 3, 'min_child_samples': 66, 'colsample_bytree': 0.6520737037218487, 'subsample': 0.5369837226967138, 'subsample_freq': 1, 'reg_alpha': 0.001877106197286048, 'reg_lambda': 0.12086070524768504, 'max_bin': 354}


In [30]:
##########--------------LAST MODEL--------------##########
model_last = LGBMRegressor(
    n_estimators=813,
    learning_rate=0.07420147247657288,
    num_leaves=47,
    max_depth=3,
    min_child_samples=66,
    colsample_bytree= 0.6520737037218487,
    subsample=0.5369837226967138,
    subsample_freq= 1,
    reg_alpha=   0.001877106197286048,
    reg_lambda= 0.12086070524768504,
    objective="regression",
    random_state=42,
    importance_type="gain",
    max_bin = 354
)
model_last.fit(X_train_v3, y_train, categorical_feature=categorical_cols_v3)

predicted_ratio_last = model_last.predict(X_val_v3)
predicted_consumption_last = predicted_ratio_last * val["yearly_baseline"]

MAPE_last = ((val["consumption"] - predicted_consumption_last).abs() / val["consumption"]).mean() * 100
print("LightGBM LAST Val MAPE (%):", MAPE_last)

LightGBM LAST Val MAPE (%): 2.919485122730832


In [31]:
predicted_ratio_train_last = model_last.predict(X_train_v3)
predicted_consumption_train_last = predicted_ratio_train_last * train["yearly_baseline"]

MAPE_train_last = ((train["consumption"] - predicted_consumption_train_last).abs() / train["consumption"]).mean() * 100
print("LightGBM Train MAPE (%):", MAPE_train_last)

LightGBM Train MAPE (%): 1.5154661339466764
